# Gold Layer Aggregation

---
## Overview
Notebook 7 of 8.

This notebook creates analysis-ready aggregate table from SDTM domains for reporting and visualization.

Run the entire notebook to load the data.

---

## Source Data

- **Sources:** silver.sdtm_dm, silver.sdtm_ae, silver.sdtm_ex
- **Layer:** Gold (Analytics)
---

## Aggregations

1. Demographic Summary
2. Exposure Summary
3. Integrated Analysis

---

## Target

Analysis tables written to **gold** schema for dashboard and reporting consumption.

---

In [0]:
from pyspark.sql.functions import col, count, sum, when

In [0]:
#global variables
CATALOG = "cluepoints_db_demo"
SCHEMA = "gold"

In [0]:
#create gold schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

In [0]:
#load the silver tables into memory
demog = spark.table("cluepoints_db_demo.silver.demog_sdtm")
ae = spark.table("cluepoints_db_demo.silver.ae_sdtm")
ex = spark.table("cluepoints_db_demo.silver.ex_sdtm")

In [0]:
#create adverse event summary - counts based on decode of serious flag and fatal outcome
ae_summary = ae.groupBy("SUBJID").agg(
    count("*").alias("TOTAL_AES"),
    sum(when(col("AESER") == "Y", 1).otherwise(0)).alias("SERIOUS_AES"),
    sum(when(col("AEOUT") == "FATAL", 1).otherwise(0)).alias("FATAL_AES")
)

In [0]:
#create exposure summary - count of total exposures
ex_summary = ex.groupBy("SUBJID").agg(
    count("*").alias("TOTAL_EXPOSURES")
)

In [0]:
#left join the summary tables onto demographic data
subject_summary = demog.join(ae_summary, "SUBJID", "left").join(ex_summary, "SUBJID", "left")

In [0]:
#fill in the zeros from the left joins
subject_summary = subject_summary.fillna(0, ["TOTAL_AES", "SERIOUS_AES", "FATAL_AES", "TOTAL_EXPOSURES"])

In [0]:
#write the final table - full overwrite for idempotent behaviour
subject_summary.write.format('delta').mode('overwrite').saveAsTable('cluepoints_db_demo.gold.subject_summary')

In [0]:
#display the table
display(spark.sql("select * from cluepoints_db_demo.gold.subject_summary limit 10"))

SUBJID,SITEID,AGE,SEX,RACE,COUNTRY,BRTHDTC,ARM,RFSTDTC,source_file,loaded_at,ingestion_id,TOTAL_AES,SERIOUS_AES,FATAL_AES,TOTAL_EXPOSURES
840414004,414,35,M,RACE1,Italy,1969-09-10,Treatment,2005-05-31,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437,1,0,0,0
840414002,414,59,F,RACE1,Italy,1945-09-30,Treatment,2005-04-29,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437,3,0,0,0
840414006,414,70,M,RACE1,Italy,1935-12-31,Placebo,2005-06-03,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437,0,0,0,0
840414001,414,58,M,RACE1,Italy,1946-09-30,Placebo,2005-04-28,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437,0,0,0,0
840414003,414,77,M,RACE1,Italy,1928-08-10,Placebo,2005-05-26,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437,6,2,0,0
840414005,414,64,F,RACE1,Italy,1940-09-05,Placebo,2005-06-03,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437,0,0,0,0
840414007,414,64,M,RACE1,Italy,1941-02-11,Placebo,2005-07-05,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437,1,0,0,0
840414099,414,99,M,RACE1,Italy,1927-02-11,Treatment,2004-02-05,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437,0,0,0,0
826290006,290,85,M,RACE1,Spain,1920-12-31,Treatment,2005-06-09,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437,1,0,0,0
826290002,290,78,F,RACE1,Spain,1925-10-25,Treatment,2004-08-10,demog.csv,2026-01-05T08:24:42.854Z,20260105_082437,14,7,0,2
